# JEDI Baseline Modeling Summary

**Project:** JEDI - Job Evaluation & Delivery Intelligence  
**Phase:** Baseline supervised modeling for `applied=yes/no`  
**Dataset:** `data/jedi_master_prepared.csv`  

This notebook documents where the project currently stands after the first baseline modeling round. It is intentionally written as a project-display notebook: it records what was tried, what scores were achieved, why the scores should be interpreted conservatively, and what the next modeling step should be.

The current baseline task is binary classification:

```text
job attributes -> predict whether I would apply
```

The baseline models do **not** use manually annotated profile-fit scores as training inputs. That was a deliberate anti-leakage decision.

## 1. Current Standing

The prepared dataset contains 331 labeled jobs:

| Label | Rows | Share |
|---|---:|---:|
| `applied=no` | 205 | 61.9% |
| `applied=yes` | 126 | 38.1% |

This is a usable first supervised dataset, but it is still small for a personal-preference model. The label is subjective: `applied=yes` means the role matched my personal career stage, role preference, location preference, salary threshold, and perceived opportunity quality.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "jedi_master_prepared.csv").exists():
    PROJECT_ROOT = Path.cwd().parents[1]

DATA_PATH = PROJECT_ROOT / "data" / "jedi_master_prepared.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

## 2. Models Tried

We trained four classical baselines:

| Model | Why it was included |
|---|---|
| Logistic Regression | Strong, interpretable baseline for TF-IDF text and one-hot features |
| Random Forest | Tree-based model to capture non-linear interactions |
| Complement Naive Bayes | Fast text-heavy sanity baseline |
| Gradient Boosting | Stronger tree ensemble and current performance candidate |

All models used raw job features, normalized categorical fields, numeric helpers, boolean location flags, and TF-IDF text. Personal fit score columns were excluded from the first-pass classifiers to avoid leakage.

In [ ]:
holdout_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": 0.910448,
        "balanced_accuracy": 0.905722,
        "precision_yes": 0.884615,
        "recall_yes": 0.884615,
        "f1_yes": 0.884615,
        "roc_auc": 0.939962,
        "average_precision": 0.917937,
        "log_loss": 0.291022,
        "brier_score": 0.079910,
        "confusion_matrix": "TN=38, FP=3, FN=3, TP=23",
    },
    {
        "model": "Random Forest",
        "accuracy": 0.910448,
        "balanced_accuracy": 0.898687,
        "precision_yes": 0.916667,
        "recall_yes": 0.846154,
        "f1_yes": 0.880000,
        "roc_auc": 0.933396,
        "average_precision": 0.899268,
        "log_loss": 0.329151,
        "brier_score": 0.092703,
        "confusion_matrix": "TN=39, FP=2, FN=4, TP=22",
    },
    {
        "model": "Complement Naive Bayes",
        "accuracy": 0.880597,
        "balanced_accuracy": 0.867261,
        "precision_yes": 0.875000,
        "recall_yes": 0.807692,
        "f1_yes": 0.840000,
        "roc_auc": 0.937148,
        "average_precision": 0.888231,
        "log_loss": 0.489864,
        "brier_score": 0.095856,
        "confusion_matrix": "TN=38, FP=3, FN=5, TP=21",
    },
    {
        "model": "Gradient Boosting",
        "accuracy": 0.925373,
        "balanced_accuracy": 0.917917,
        "precision_yes": 0.920000,
        "recall_yes": 0.884615,
        "f1_yes": 0.901961,
        "roc_auc": 0.918386,
        "average_precision": 0.898197,
        "log_loss": 0.299486,
        "brier_score": 0.077915,
        "confusion_matrix": "TN=39, FP=2, FN=3, TP=23",
    },
])

display(holdout_results.sort_values("f1_yes", ascending=False))

## 3. K-Fold Cross-Validation Results

The held-out test split is useful, but the cross-validation numbers are the more honest estimate because the dataset is small. A single lucky or unlucky split can distort results.

Cross-validation tells us: if we repeatedly train and test on different folds, how stable is the model?

In [ ]:
cv_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy_mean": 0.873225,
        "accuracy_std": 0.029040,
        "balanced_accuracy_mean": 0.862889,
        "f1_mean": 0.829553,
        "f1_std": 0.043536,
        "precision_mean": 0.846074,
        "recall_mean": 0.818462,
        "roc_auc_mean": 0.922904,
        "average_precision_mean": 0.897248,
    },
    {
        "model": "Random Forest",
        "accuracy_mean": 0.879195,
        "accuracy_std": 0.025122,
        "balanced_accuracy_mean": 0.858248,
        "f1_mean": 0.827723,
        "f1_std": 0.040892,
        "precision_mean": 0.898099,
        "recall_mean": 0.770154,
        "roc_auc_mean": 0.923722,
        "average_precision_mean": 0.902026,
    },
    {
        "model": "Complement Naive Bayes",
        "accuracy_mean": 0.842877,
        "accuracy_std": 0.015654,
        "balanced_accuracy_mean": 0.832101,
        "f1_mean": 0.791423,
        "f1_std": 0.022059,
        "precision_mean": 0.804175,
        "recall_mean": 0.786154,
        "roc_auc_mean": 0.909598,
        "average_precision_mean": 0.883163,
    },
    {
        "model": "Gradient Boosting",
        "accuracy_mean": 0.885391,
        "accuracy_std": 0.035630,
        "balanced_accuracy_mean": 0.864841,
        "f1_mean": 0.838097,
        "f1_std": 0.048996,
        "precision_mean": 0.910498,
        "recall_mean": 0.778462,
        "roc_auc_mean": 0.923490,
        "average_precision_mean": 0.890714,
    },
])

display(cv_results.sort_values("f1_mean", ascending=False))

## 4. Hyperparameter And Tuning Notes

The baseline phase was not meant to exhaustively tune every possible model. The goal was to identify a reliable baseline and decide whether more complex modeling is justified.

| Model | Tuning / parameter decision | Resulting decision |
|---|---|---|
| Logistic Regression | Tried a `C` grid: `0.05, 0.10, 0.50, 1.00, 2.00, 10.00` | `C=2.0` had the best CV F1 around `0.837`; default `C=1.0` was close and stable |
| Random Forest | Used 400 trees, `min_samples_leaf=2`, and balanced subsample weights | Strong precision, weaker recall; useful as a second-opinion model, not the main baseline |
| Complement Naive Bayes | Used `alpha=0.5`; major gains are not expected from heavy tuning | Kept as a text sanity baseline |
| Gradient Boosting | Used shallow trees, low learning rate, subsampling, and balanced sample weights | Best held-out F1, but only slightly better than logistic in CV |

Smart stopping decision: we did not over-invest in Naive Bayes or Random Forest tuning because their role is baseline comparison, not final production modeling. The next real improvement is more likely to come from profile-fit score modeling and better validation than from squeezing another small percentage out of these baselines.

In [ ]:
logistic_c_tuning = pd.DataFrame([
    {"C": 10.00, "f1_mean": 0.831121, "roc_auc_mean": 0.921411, "average_precision_mean": 0.902724, "balanced_accuracy_mean": 0.864143},
    {"C": 2.00, "f1_mean": 0.836888, "roc_auc_mean": 0.922326, "average_precision_mean": 0.899574, "balanced_accuracy_mean": 0.869328},
    {"C": 1.00, "f1_mean": 0.829553, "roc_auc_mean": 0.922904, "average_precision_mean": 0.897248, "balanced_accuracy_mean": 0.862889},
    {"C": 0.50, "f1_mean": 0.823009, "roc_auc_mean": 0.922717, "average_precision_mean": 0.896722, "balanced_accuracy_mean": 0.858011},
    {"C": 0.10, "f1_mean": 0.823832, "roc_auc_mean": 0.921208, "average_precision_mean": 0.894041, "balanced_accuracy_mean": 0.857704},
    {"C": 0.05, "f1_mean": 0.820939, "roc_auc_mean": 0.918116, "average_precision_mean": 0.889142, "balanced_accuracy_mean": 0.855265},
])

display(logistic_c_tuning.sort_values("f1_mean", ascending=False))

## 5. Prediction Files From All Models

Each baseline notebook exported test-set predictions. These files are useful for manual error analysis: compare high-confidence false positives, false negatives, and jobs where models disagree.

In [ ]:
prediction_files = {
    "logistic_regression": PROJECT_ROOT / "data" / "jedi_logistic_regression_test_predictions.csv",
    "random_forest": PROJECT_ROOT / "data" / "jedi_random_forest_test_predictions.csv",
    "naive_bayes": PROJECT_ROOT / "data" / "jedi_naive_bayes_test_predictions.csv",
    "gradient_boosting": PROJECT_ROOT / "data" / "jedi_gradient_boosting_test_predictions.csv",
}

prediction_summaries = []
for model_name, path in prediction_files.items():
    preds = pd.read_csv(path)
    prediction_summaries.append({
        "model": model_name,
        "prediction_file": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(preds),
        "mean_applied_probability": preds["applied_probability"].mean(),
        "max_applied_probability": preds["applied_probability"].max(),
    })

display(pd.DataFrame(prediction_summaries))

## 6. Why Scores Are Not Higher Yet

The cross-validation F1 range is roughly `0.79` to `0.84`. That is not bad for this stage, but it is not final-production quality. The main blockers are dataset maturity and target complexity:

1. **Small dataset:** 331 rows is limited for a personal preference model with text, categorical fields, location, salary, company, and skills.
2. **Subjective target:** `applied` is not a universal quality label. It is my personal action decision.
3. **Profile scores excluded:** the strongest human preference columns were excluded from classifier inputs to avoid leakage.
4. **Salary saturation:** most scraped jobs are already good-paying roles. Since my `12 LPA` threshold maps to a high preference score, salary has low variation and weak predictive value right now.
5. **Limited numeric signal:** most useful information is textual or categorical, not numeric.
6. **Sparse segments:** some seniority, role-family, city, and employment-type categories have too few examples.
7. **Duplicate risk:** similar jobs can appear across sources. Future validation should use `near_duplicate_key` grouped splits.
8. **Borderline examples are underrepresented:** the model needs more jobs that are hard to decide, not only obvious yes/no examples.

## 7. Decisions Made In The Baseline Phase

The baseline phase made several smart modeling decisions:

- Kept `applied` as the first binary classification target.
- Excluded true profile-fit scores from classifier features to avoid leakage.
- Dropped `source_platform` and `source_type` so the model learns job preference instead of scraper/source bias.
- Used TF-IDF for role title and skills text instead of forcing text into fragile manual categories.
- Used one-hot encoding for standardized enums such as role family, seniority, work mode, employment type, and location bucket.
- Used class balancing because `applied=yes` is the minority class.
- Used k-fold cross-validation to avoid trusting one lucky split.
- Treated logistic regression as the official baseline because it is interpretable, stable, and nearly tied with more complex models.
- Treated gradient boosting as the current performance candidate, not as the final answer yet.
- Stopped over-tuning weaker baselines and moved toward a higher-value next experiment.

## 8. Next Step: Profile-Fit Score Stacking

The next experiment should test whether predicted profile-fit scores improve the final `applied` classifier.

The correct architecture is:

```text
raw job fields
    -> score models predict profile-fit scores
    -> predicted scores are appended as features
    -> applied classifier predicts yes/no
```

Important rule:

```text
Use predicted profile scores, not true manually annotated profile scores, when training the final applied classifier.
```

Recommended experiments:

| Experiment | Meaning |
|---|---|
| A. Raw-only classifier | Current honest baseline |
| B. Raw + true profile scores | Oracle/leakage test; shows maximum possible value of score features |
| C. Raw + out-of-fold predicted profile scores | Realistic stacked model; this is the candidate architecture |

Primary score targets for the next notebook:

- `overall_preference_score`
- `role_title_preference_score`
- `role_family_preference_score`
- `seniority_profile_fit_score`
- `location_profile_fit_score`
- `skills_profile_fit_score`

This is a better next step than more classical baseline tuning because it uses the richer supervision already present in the annotation schema.